# RV32I 5-Stage Pipeline CPU — PYNQ-ZU 板上整合測試
依序執行所有 cell。若重新上傳 bitstream，請先 **Kernel → Restart**。

- 舊版 bitstream（4 個暫存器）：跑 T1~T3
- 新版（含 trace buffer）：另外跑 T4~T6 逐 cycle 比對 ISA 黃金模型

In [ ]:
# ① 載入 Overlay 與 MMIO（改檔後請先 Kernel → Restart，避免 PYNQ 使用舊 bitstream 快取）
import time, json
from pynq import Overlay, MMIO

BIT  = "/home/xilinx/jupyter_notebooks/design_1_wrapper.bit"   # 同資料夾需有同名 .hwh
BASE = 0x80000000
SIZE = 0x10000

ol   = Overlay(BIT)
mmio = MMIO(BASE, SIZE)
print("Overlay loaded:", ol.is_loaded() if hasattr(ol, "is_loaded") else True)
print("CPU IP        :", ol.ip_dict["rv32i_pipelined_cpu_0"]["type"])


In [ ]:
# ② RV32I 黃金模型（ISA 層級）與預設測試程式（= sim/program.hex = imem.v 內建程式）
# 最小 RV32I ISA 模擬器：輸出預期的 (rd, value) 寫回序列（rd!=0）
M32 = 0xFFFFFFFF
def sx(v, b): return v - (1 << b) if v & (1 << (b - 1)) else v
def load_hex(path):
    return [int(l.strip(), 16) for l in open(path) if l.strip() and not l.startswith("//")]
def golden(prog, nmax=2000, stop_pc=None):
    mem = {}; x = [0] * 32; pc = 0; wb = []
    n = len(prog)
    for _ in range(nmax):
        ins = prog[pc // 4] if pc // 4 < n else 0x13
        op = ins & 0x7F; rd = (ins >> 7) & 31; f3 = (ins >> 12) & 7
        rs1 = (ins >> 15) & 31; rs2 = (ins >> 20) & 31; f7 = ins >> 25
        npc = pc + 4; val = None
        immi = sx(ins >> 20, 12)
        if op == 0x13:
            a = x[rs1]
            val = {0: (a + immi), 4: a ^ (immi & M32), 6: a | (immi & M32), 7: a & (immi & M32),
                   2: int(sx(a, 32) < immi), 3: int(a < (immi & M32)),
                   1: a << (immi & 31), 5: (sx(a, 32) >> (immi & 31)) if f7 == 0x20 else a >> (immi & 31)}[f3] & M32
        elif op == 0x33:
            a, b = x[rs1], x[rs2]
            val = {(0, 0): a + b, (0, 0x20): a - b, (7, 0): a & b, (6, 0): a | b, (4, 0): a ^ b}[(f3, f7)] & M32
        elif op == 0x37: val = ins & 0xFFFFF000
        elif op == 0x17: val = (pc + (ins & 0xFFFFF000)) & M32
        elif op == 0x03:
            val = mem.get((x[rs1] + immi) & M32 & ~3, 0)
        elif op == 0x23:
            imm = sx(((ins >> 25) << 5) | ((ins >> 7) & 31), 12)
            mem[(x[rs1] + imm) & M32 & ~3] = x[rs2]
        elif op == 0x63:
            imm = sx(((ins >> 31) << 12) | (((ins >> 7) & 1) << 11) | (((ins >> 25) & 0x3F) << 5) | (((ins >> 8) & 0xF) << 1), 13)
            a, b = x[rs1], x[rs2]
            t = {0: a == b, 1: a != b, 4: sx(a, 32) < sx(b, 32), 5: sx(a, 32) >= sx(b, 32), 6: a < b, 7: a >= b}[f3]
            if t: npc = pc + imm
        elif op == 0x6F:
            imm = sx(((ins >> 31) << 20) | (((ins >> 12) & 0xFF) << 12) | (((ins >> 20) & 1) << 11) | (((ins >> 21) & 0x3FF) << 1), 21)
            val = pc + 4; npc = pc + imm
        elif op == 0x67:
            val = pc + 4; npc = (x[rs1] + immi) & ~1 & M32
        if val is not None and rd:
            x[rd] = val & M32; wb.append((rd, val & M32))
        if npc == pc: break           # spin（自我跳躍）
        pc = npc & M32
    return wb, x, pc


PROGRAM = [
    0x00500093, 0x00a00113, 0x002081b3, 0x00500213,
    0x00320293, 0x00528b93, 0x00100313, 0x00130393,
    0x00138413, 0x00000593, 0x02a00493, 0x0095a023,
    0x0005a503, 0x00950633, 0x00000493, 0x00050493,
    0x00000693, 0x00100793, 0x00100813, 0x01078463,
    0x06300693, 0x00700713, 0x00100913, 0x00200993,
    0x01391663, 0x00500893, 0x00900893, 0x00900893,
    0x00000013, 0x00000013, 0x00000a93, 0x00c000ef,
    0x00500a93, 0x00700a93, 0x06300a13, 0x0080006f,
    0x00500a93, 0x12345b37, 0x0040006f, 0x0000006f
]
EXP_WB, EXP_REGS, EXP_END_PC = golden(PROGRAM)
print("黃金模型：%d 筆寫回，最終 spin PC = 0x%X" % (len(EXP_WB), EXP_END_PC))


In [ ]:
# ③ 暫存器對應與測試工具
CTRL, PC, WBD, DMEM       = 0x00, 0x04, 0x08, 0x0C
T_STATUS, T_IDX, T_PC, T_WB, T_META = 0x10, 0x14, 0x18, 0x1C, 0x20   # trace buffer（新版 bitstream）

def cpu_reset():            mmio.write(CTRL, 0x0)
def cpu_run():              mmio.write(CTRL, 0x1)
def arm_and_run():          mmio.write(CTRL, 0x2); mmio.write(CTRL, 0x1)   # reset+arm → run

def read_trace(n=256):
    rows = []
    for i in range(n):
        mmio.write(T_IDX, i)
        pc, wd, meta = mmio.read(T_PC), mmio.read(T_WB), mmio.read(T_META)
        rows.append((i, pc, wd, (meta >> 1) & 31, meta & 1))     # (cycle, pc, wb_data, wb_rd, wb_we)
    return rows

def wb_events(rows):
    return [(r[3], r[2]) for r in rows if r[4] == 1 and r[3] != 0]

RESULTS = []                 # (名稱, 通過?, 說明)
def check(name, ok, detail=""):
    RESULTS.append((name, bool(ok), detail))
    print(("[PASS] " if ok else "[FAIL] ") + name + (("  —  " + detail) if detail else ""))
    return ok


In [ ]:
# ④ 基本測試：reset / run / 重跑（不需要 trace，舊版與新版 bitstream 都能跑）
SPIN_PCS = {EXP_END_PC, EXP_END_PC + 4, EXP_END_PC + 8}     # spin 迴圈附近的 PC（含 flush 時的預取）

# T1 reset 狀態
cpu_reset(); time.sleep(0.001)
r0, pc0 = mmio.read(CTRL), mmio.read(PC)
check("T1 reset：rst_n=0 且 PC=0", (r0 & 1) == 0 and pc0 == 0, "reg0=0x%X pc=0x%X" % (r0, pc0))

# T2 run 後 PC 落在 spin 附近
cpu_run(); time.sleep(0.01)
r0 = mmio.read(CTRL)
pcs = sorted({mmio.read(PC) for _ in range(100)})
check("T2a run：rst_n 寫入成功（reg0[0]=1）", r0 & 1 == 1, "reg0=0x%X" % r0)
check("T2b run：PC 停在 spin 迴圈附近", set(pcs) <= SPIN_PCS and len(pcs) > 0, "PC 集合=" + str([hex(p) for p in pcs]))
wbs  = {mmio.read(WBD)  for _ in range(200)}
dms  = {mmio.read(DMEM) for _ in range(200)}
check("T2c run：取樣到 jal 的返回位址 0x%X" % (EXP_END_PC + 4), (EXP_END_PC + 4) in wbs, "wb_data 取樣=" + str(sorted(hex(x) for x in wbs)))
check("T2d run：取樣到 sw/lw 存入的 42", 42 in dms, "dmem_rdata 取樣=" + str(sorted(hex(x) for x in dms)))

# T3 reset 回到起點、重跑可重複
cpu_reset(); time.sleep(0.001)
check("T3a 再次 reset：PC 回到 0", mmio.read(PC) == 0)
ok = True
for k in range(5):
    cpu_reset(); time.sleep(0.001); cpu_run(); time.sleep(0.005)
    p = {mmio.read(PC) for _ in range(30)}
    ok &= p <= SPIN_PCS
check("T3b 重複 reset/run 5 次結果一致", ok)


In [ ]:
# ⑤ 逐 cycle 驗證（需要新版 trace buffer bitstream；舊版會自動略過）
arm_and_run(); time.sleep(0.01)
st = mmio.read(T_STATUS)
HAS_TRACE = bool(st & 0x80000000)
if not HAS_TRACE:
    print("[SKIP] 偵測不到 trace buffer（status=0x%X）：目前載入的是舊版 bitstream，略過 T4~T6" % st)
else:
    check("T4a trace 已填滿 256 筆", (st & 0x1FF) == 256, "status=0x%X" % st)
    rows1 = read_trace()
    # T4 寫回序列與黃金模型逐筆比對
    got = wb_events(rows1)
    first_bad = next((i for i, (e, g) in enumerate(zip(EXP_WB, got)) if e != g), None)
    check("T4b 寫回序列與 ISA 黃金模型一致（%d 筆）" % len(EXP_WB),
          got == EXP_WB,
          "" if got == EXP_WB else ("第 %s 筆不同：預期 %s 實際 %s；總筆數 預期%d 實際%d" %
              (first_bad, EXP_WB[first_bad] if first_bad is not None and first_bad < len(EXP_WB) else None,
               got[first_bad] if first_bad is not None and first_bad < len(got) else None, len(EXP_WB), len(got))))
    # T4c 起始行為
    check("T4c reset 放開後第 0 個 cycle PC=0", rows1[0][1] == 0, "pc[0]=0x%X" % rows1[0][1])
    # T4d PC 永遠 4 byte 對齊，且在程式範圍或 spin 附近
    pcs_all = [r[1] for r in rows1]
    check("T4d PC 皆 4-byte 對齊且不跑出程式範圍",
          all(p % 4 == 0 for p in pcs_all) and max(pcs_all) <= EXP_END_PC + 8, "max pc=0x%X" % max(pcs_all))
    # T4e 最終進入 spin
    check("T4e 最後 16 個 cycle 都在 spin 迴圈內", set(pcs_all[-16:]) <= SPIN_PCS)
    # T4f 最終暫存器內容（由寫回序列重建）與黃金模型一致
    regs = [0] * 32
    for rd, v in got: regs[rd] = v
    diff = [(i, regs[i], EXP_REGS[i]) for i in range(1, 32) if regs[i] != EXP_REGS[i]]
    check("T4f 由 trace 重建的暫存器 x1~x31 與黃金模型一致", not diff, "差異=" + str(diff[:4]))

    # T5 再跑一次，trace 必須完全相同（決定性）
    arm_and_run(); time.sleep(0.01)
    rows2 = read_trace()
    check("T5 重跑後逐 cycle trace 完全相同", rows1 == rows2)

    # T6 執行中途 reset：PC 回 0，並能重新開始
    cpu_reset(); time.sleep(0.001)
    check("T6a 執行後 reset：PC 回到 0", mmio.read(PC) == 0)
    arm_and_run(); time.sleep(0.01)
    rows3 = read_trace()
    check("T6b reset 後重新執行，trace 與第一次相同", rows3 == rows1)


In [ ]:
# ⑥ 結果總表、前 40 個 cycle 的 trace、並存成報告檔
print("=" * 64)
for name, ok, detail in RESULTS:
    print(("PASS  " if ok else "FAIL  ") + name)
n_pass = sum(1 for _, ok, _ in RESULTS if ok)
print("=" * 64)
print("總計：%d / %d 通過  →  %s" % (n_pass, len(RESULTS), "ALL PASS ✅" if n_pass == len(RESULTS) else "有失敗項目 ❌"))

if HAS_TRACE:
    print("\n前 40 個 cycle（cyc / pc / wb_data / wb_rd / wb_we）：")
    for r in rows1[:40]:
        print("%3d  0x%08X  0x%08X  x%-2d  %d" % r)

report = {"time": time.strftime("%Y-%m-%d %H:%M:%S"),
          "results": [{"test": n, "pass": ok, "detail": d} for n, ok, d in RESULTS],
          "trace": rows1 if HAS_TRACE else None}
with open("/home/xilinx/jupyter_notebooks/cpu_test_report.json", "w") as f:
    json.dump(report, f, indent=1)
print("\n報告已存到 cpu_test_report.json")
